# Introduction to Neural Networks. Multi-Layered Perceptron

**AI1 · Neural Networks** — [AI Olympiad Curriculum](https://abzlitochka.github.io/ai-olympiad-curriculum)

~15 min

## Introduction

In the previous section, you learned about the simplest neural network model - one-layered perceptron, a linear two-class classification model.

In this section we will extend this model into a more flexible framework, allowing us to:

* perform **multi-class classification** in addition to two-class
* solve **regression problems** in addition to classification
* separate classes that are not linearly separable

We will also develop our own modular framework in Python that will allow us to construct different neural network architectures.



## Formalization of Machine Learning

Let's start with formalizing the Machine Learning problem. Suppose we have a training dataset **X** with labels **Y**, and we need to build a model *f* that will make most accurate predictions. The quality of predictions is measured by **Loss function** ℒ. The following loss functions are often used:

* For regression problem, when we need to predict a number, we can use **absolute error** ∑<sub>i</sub>|f(x<sup>(i)</sup>)-y<sup>(i)</sup>|, or **squared error** ∑<sub>i</sub>(f(x<sup>(i)</sup>)-y<sup>(i)</sup>)<sup>2</sup>
* For classification, we use **0-1 loss** (which is essentially the same as **accuracy** of the model), or **logistic loss**.

For one-level perceptron, function *f* was defined as a linear function *f(x)=wx+b* (here *w* is the weight matrix, *x* is the vector of input features, and *b* is bias vector). For different neural network architectures, this function can take more complex form.

> In the case of classification, it is often desirable to get probabilities of corresponding classes as network output. To convert arbitrary numbers to probabilities (eg. to normalize the output), we often use **softmax** function σ, and the function *f* becomes *f(x)=σ(wx+b)*

In the definition of *f* above, *w* and *b* are called **parameters** θ=⟨*w,b*⟩. Given the dataset ⟨**X**,**Y**⟩, we can compute an overall error on the whole dataset as a function of parameters θ.

> ✅ **The goal of neural network training is to minimize the error by varying parameters θ**



## Gradient Descent Optimization

There is a well-known method of function optimization called **gradient descent**. The idea is that we can compute a derivative (in multi-dimensional case called **gradient**) of loss function with respect to parameters, and vary parameters in such a way that the error would decrease. This can be formalized as follows:

* Initialize parameters by some random values w<sup>(0)</sup>, b<sup>(0)</sup>
* Repeat the following step many times:
    - w<sup>(i+1)</sup> = w<sup>(i)</sup>-η∂ℒ/∂w
    - b<sup>(i+1)</sup> = b<sup>(i)</sup>-η∂ℒ/∂b

During training, the optimization steps are supposed to be calculated considering the whole dataset (remember that loss is calculated as a sum through all training samples). However, in real life we take small portions of the dataset called **minibatches**, and calculate gradients based on a subset of data. Because subset is taken randomly each time, such method is called **stochastic gradient descent** (SGD).



## Multi-Layered Perceptrons and Backpropagation

One-layer network, as we have seen above, is capable of classifying linearly separable classes. To build a richer model, we can combine several layers of the network. Mathematically it would mean that the function *f* would have a more complex form, and will be computed in several steps:
* z<sub>1</sub>=w<sub>1</sub>x+b<sub>1</sub>
* z<sub>2</sub>=w<sub>2</sub>α(z<sub>1</sub>)+b<sub>2</sub>
* f = σ(z<sub>2</sub>)

Here, α is a **non-linear activation function**, σ is a softmax function, and parameters θ=<*w<sub>1</sub>,b<sub>1</sub>,w<sub>2</sub>,b<sub>2</sub>*>.

The gradient descent algorithm would remain the same, but it would be more difficult to calculate gradients. Given the chain differentiation rule, we can calculate derivatives as:

* ∂ℒ/∂w<sub>2</sub> = (∂ℒ/∂σ)(∂σ/∂z<sub>2</sub>)(∂z<sub>2</sub>/∂w<sub>2</sub>)
* ∂ℒ/∂w<sub>1</sub> = (∂ℒ/∂σ)(∂σ/∂z<sub>2</sub>)(∂z<sub>2</sub>/∂α)(∂α/∂z<sub>1</sub>)(∂z<sub>1</sub>/∂w<sub>1</sub>)

> ✅ The chain differentiation rule is used to calculate derivatives of the loss function with respect to parameters.

Note that the left-most part of all those expressions is the same, and thus we can effectively calculate derivatives starting from the loss function and going "backwards" through the computational graph. Thus the method of training a multi-layered perceptron is called **backpropagation**, or 'backprop'.

![compute graph](ms-assets/redrawn/compute-graph-grad.svg)

> TODO: image citation

> ✅ We will cover backprop in much more detail in our notebook example.



## Conclusion

In this lesson, we have built our own neural network library, and we have used it for a simple two-dimensional classification task.



## Challenge

In the accompanying notebook, you will implement your own framework for building and training multi-layered perceptrons. You will be able to see in detail how modern neural networks operate.

Proceed to the [OwnFramework](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/3-NeuralNetworks/04-OwnFramework/OwnFramework.ipynb) notebook and work through it.



## Review & Self Study

Backpropagation is a common algorithm used in AI and ML, worth studying [in more detail](https://wikipedia.org/wiki/Backpropagation)



## Assignment

In this lab, you are asked to use the framework you constructed in this lesson to solve MNIST handwritten digit classification.

* [Instructions](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/3-NeuralNetworks/04-OwnFramework/lab/README.md)
* [Notebook](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/3-NeuralNetworks/04-OwnFramework/lab/MyFW_MNIST.ipynb)



---

## ✅ Self-check tasks

Implement the function in the 'your code' cell, then run the check cell — it runs randomized tests and shows the first mismatch. Run the helper cell below once first.

In [ ]:
# ── task-checking helper cell — run once ──
import copy
import numpy as np
import math, statistics, itertools, collections, functools, heapq, random

try:
    import sys; sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

_L = {"noFn":"❓ Define a function `{e}` in the solution cell and run it","runErr":"❌ Test {i}: {t}: {m}","wrong":"❌ Test {i}: {r}","badRef":"⚠️ Check harness broke (generator/reference): {t}: {m}","inL":"Input","expL":"Expected","gotL":"Got","ok":"✅ Correct! All {n} tests passed."}

def _nb_copy_kw(kw):
    return {k: (v.copy() if isinstance(v, np.ndarray) else copy.deepcopy(v)) for k, v in kw.items()}

def _nb_jsafe(x, depth=0):
    if depth > 6: return "…"
    if isinstance(x, np.ndarray): return _nb_jsafe(x.tolist(), depth + 1)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.floating): return float(x)
    if isinstance(x, np.bool_): return bool(x)
    if isinstance(x, float): return round(x, 6) if math.isfinite(x) else str(x)
    if isinstance(x, (list, tuple)): return [_nb_jsafe(v, depth + 1) for v in x]
    if isinstance(x, dict): return {str(k): _nb_jsafe(v, depth + 1) for k, v in x.items()}
    if isinstance(x, (int, str, bool)) or x is None: return x
    return repr(x)

def _nb_cmp(exp, got, path=""):
    if isinstance(exp, np.ndarray) or isinstance(got, np.ndarray):
        e, g = np.asarray(exp), np.asarray(got)
        if e.shape != g.shape:
            return f"shape {tuple(g.shape)} != expected {tuple(e.shape)}"
        try:
            if np.issubdtype(e.dtype, np.number) or np.issubdtype(g.dtype, np.number):
                ef, gf = e.astype(float), g.astype(float)
                close = np.isclose(gf, ef, atol=1e-6, rtol=1e-5, equal_nan=True)
                if not bool(close.all()):
                    i = tuple(int(v) for v in np.argwhere(~close)[0])
                    return f"{path}first mismatch at index {i}: got {gf[i]:.6g} vs expected {ef[i]:.6g}"
            elif not np.array_equal(g, e):
                return f"{path}values differ"
        except Exception as ex:
            return f"{path}cannot compare: {ex}"
        return None
    if isinstance(exp, (int, float)) and not isinstance(exp, bool) and isinstance(got, (int, float)) and not isinstance(got, bool):
        if not np.isclose(float(exp), float(got), atol=1e-6, rtol=1e-5):
            return f"{path}{got} != expected {exp}"
        return None
    if isinstance(exp, (list, tuple)) and isinstance(got, (list, tuple)):
        if len(exp) != len(got):
            return f"{path}length {len(got)} != expected {len(exp)}"
        for i, (a, b) in enumerate(zip(exp, got)):
            r = _nb_cmp(a, b, path + f"[{i}].")
            if r: return r
        return None
    if isinstance(exp, dict) and isinstance(got, dict):
        if set(exp) != set(got):
            return f"{path}keys differ"
        for k in exp:
            r = _nb_cmp(exp[k], got[k], path + f"[{k!r}].")
            if r: return r
        return None
    if exp != got:
        return f"{path}{got!r} != expected {exp!r}"
    return None

def _check_task(g, ref_src, entry, n=20, seed=0):
    """g — globals() ноутбука; ref_src — gen() + эталонное решение."""
    fn = g.get(entry)
    if not callable(fn):
        print(_L["noFn"].format(e=entry)); return
    ns = {"np": np, "math": math, "statistics": statistics, "itertools": itertools,
          "collections": collections, "functools": functools, "heapq": heapq, "random": random}
    try:
        exec(ref_src, ns)
        gen, ref = ns["gen"], ns[entry]
    except Exception as e:
        print(_L["badRef"].format(t=type(e).__name__, m=e)); return
    rng = np.random.default_rng(seed)
    for i in range(n):
        try:
            kw = gen(rng)
        except Exception as e:
            print(_L["badRef"].format(t=type(e).__name__, m=e)); return
        try:
            got = fn(**_nb_copy_kw(kw))
        except Exception as e:
            print(_L["runErr"].format(i=i + 1, t=type(e).__name__, m=e)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))); return
        exp = ref(**_nb_copy_kw(kw))
        r = _nb_cmp(exp, got)
        if r:
            print(_L["wrong"].format(i=i + 1, r=r)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))
                  + "\n" + _L["expL"] + ": " + repr(_nb_jsafe(exp))
                  + "\n" + _L["gotL"] + ": " + repr(_nb_jsafe(got))); return
    print(_L["ok"].format(n=n))

### Task 1 — Numerically stable softmax

●○○ · ~8 min

> Softmax turns a vector of logits into probabilities: p_i = e^{z_i} / sum_j e^{z_j}. Naively exponentiating large logits overflows to inf (exp blows up around +710 in float64, +89 in float32); the standard fix is subtracting the row max first, which leaves the result mathematically identical.

Implement `softmax(Z)` for a 2-D matrix: apply softmax **row-wise** (axis=1). Must stay correct for large logits (e.g. ±60) — no NaN/inf allowed.

<details><summary>Hint</summary>

M = Z.max(axis=1, keepdims=True); E = np.exp(Z - M); return E / E.sum(axis=1, keepdims=True).

</details>


In [ ]:
import numpy as np

def softmax(Z):
    Z = np.asarray(Z, dtype=float)
    # your code
    return Z

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    n = int(rng.integers(2, 10)); k = int(rng.integers(2, 8))\n    Z = rng.normal(0, 20, (n, k))\n    if rng.random() < 0.4:\n        Z = Z + float(rng.uniform(900, 1400))   # exp overflows without max-subtraction\n    return {\"Z\": Z}\n\ndef softmax(Z):\n    Z = np.asarray(Z, dtype=float)\n    Z = Z - Z.max(axis=1, keepdims=True)\n    E = np.exp(Z)\n    return E / E.sum(axis=1, keepdims=True)", "softmax", 20)

### Task 2 — Cross-entropy from logits

●●○ · ~12 min

> Cross-entropy loss for the true class is -log p_y, averaged over the batch. Never compute softmax and then log separately — it loses precision and can produce log(0). Use the log-sum-exp identity: CE = log sum_j e^{z_j} - z_y, with max subtraction for stability.

Implement `cross_entropy(logits, y)`: `logits` is an (n, k) matrix of raw scores, `y` an (n,) integer array of true class indices. Return the **mean** cross-entropy over the batch as a float, computed in a numerically stable way.

<details><summary>Hint</summary>

Shift by the row max m, then lse = m + log(exp(Z - m).sum(axis=1)). The loss per row is lse - z_y; index z_y with np.arange(n) and y.

</details>


In [ ]:
import numpy as np

def cross_entropy(logits, y):
    logits = np.asarray(logits, dtype=float)
    # your code
    return 0.0

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    n = int(rng.integers(2, 12)); k = int(rng.integers(2, 8))\n    y = rng.integers(0, k, n)\n    logits = rng.normal(0, 4, (n, k))\n    if rng.random() < 0.4:\n        # z_y ~800 below the row max: naive -log(softmax) underflows to inf;\n        # a stable log-sum-exp stays finite\n        logits[np.arange(n), y] -= float(rng.uniform(800, 900))\n    return {\"logits\": logits, \"y\": y}\n\ndef cross_entropy(logits, y):\n    Z = np.asarray(logits, dtype=float)\n    y = np.asarray(y)\n    m = Z.max(axis=1)\n    lse = m + np.log(np.exp(Z - m[:, None]).sum(axis=1))\n    return float(np.mean(lse - Z[np.arange(len(y)), y]))", "cross_entropy", 20)

### Task 3 — Label smoothing of one-hot targets

●○○ · ~8 min

> Label smoothing replaces a hard one-hot target (1 on the true class, 0 elsewhere) with a soft target: the true class gets 1 − eps + eps/k and every other class gets eps/k. It stops a classifier from driving its logits to infinity to satisfy an overconfident target, which makes it a simple and effective regularizer for cross-entropy training. With eps = 0 the formula collapses back to ordinary one-hot — a good sanity check.

Implement `label_smooth(y, k, eps)` → an (n, k) float matrix where row i has `1 - eps + eps/k` at position `y[i]` and `eps/k` everywhere else.

```
y=[1], k=3, eps=0.3 → [[0.1, 0.8, 0.1]]
```

<details><summary>Hint</summary>

Start from np.eye(k)[y] for the hard part, then blend: (1 - eps) * onehot + eps / k.

</details>


In [ ]:
import numpy as np

def label_smooth(y, k, eps):
    y = np.asarray(y)
    # your code
    return np.zeros((len(y), k))

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    n, k = int(rng.integers(3, 30)), int(rng.integers(2, 7))\n    return {\"y\": rng.integers(0, k - 1, n) if rng.random() < 0.35 else rng.integers(0, k, n),\n            \"k\": k,\n            \"eps\": float(rng.choice([0.0, 0.05, 0.1, 0.2, 0.3]))}\n\ndef label_smooth(y, k, eps):\n    y = np.asarray(y)\n    return (1.0 - eps) * np.eye(k)[y] + eps / k", "label_smooth", 20)

### Task 4 — Linear layer backward

●●○ · ~12 min

> The upstream gradient dY tells you how the loss changes when each output changes. Apply the chain rule to one weight first: in y = x*w + b, an input x = 3 and upstream gradient 2 give a weight-gradient contribution of 6 and a bias-gradient contribution of 2. A shared weight affects every sample, so add its contributions across the batch. Translate that reasoning to arrays: the weight gradient must have shape (d, k), and the bias gradient shape (k,). Do not divide by the batch size again; any averaging in the loss is already included in dY.

Implement `linear_backward(X, dY)`: `X` is the (n, d) input of a linear layer Y = X @ W + b, and `dY` is the (n, k) upstream gradient. Return a tuple `(dW, db)` — gradients w.r.t. W (shape (d, k)) and b (shape (k,)).

<details><summary>Hint</summary>

dW = X.T @ dY; db = dY.sum(axis=0).

</details>


In [ ]:
import numpy as np

def linear_backward(X, dY):
    X = np.asarray(X, dtype=float)
    dY = np.asarray(dY, dtype=float)
    # your code
    return np.zeros((X.shape[1], dY.shape[1])), np.zeros(dY.shape[1])

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    n = int(rng.integers(2, 9)); d = int(rng.integers(2, 7)); k = int(rng.integers(2, 6))\n    return {\"X\": rng.normal(0, 1, (n, d)), \"dY\": rng.normal(0, 1, (n, k))}\n\ndef linear_backward(X, dY):\n    X = np.asarray(X, dtype=float)\n    dY = np.asarray(dY, dtype=float)\n    return X.T @ dY, dY.sum(axis=0)", "linear_backward", 20)

### Task 5 — SGD with momentum

●●○ · ~10 min

> Momentum keeps a velocity vector that accumulates a moving sum of past gradients (PyTorch convention): v <- mu * v + g, then w <- w - lr * v. It damps oscillations across a valley's walls and accelerates along directions where the gradient is consistent.

Implement `momentum_step(w, g, v, lr, mu)`: perform one SGD-with-momentum update using the convention `v_new = mu * v + g`, `w_new = w - lr * v_new`. Return the tuple `(w_new, v_new)`.

<details><summary>Hint</summary>

Update v first, then use the NEW v for the weight step: v_new = mu * v + g; w_new = w - lr * v_new.

</details>


In [ ]:
import numpy as np

def momentum_step(w, g, v, lr, mu):
    w = np.asarray(w, dtype=float)
    g = np.asarray(g, dtype=float)
    v = np.asarray(v, dtype=float)
    # your code — naive SGD below is wrong, momentum must use v
    return w - lr * g, v

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    d = int(rng.integers(2, 8))\n    return {\"w\": rng.normal(0, 1, d), \"g\": rng.normal(0, 1, d),\n            \"v\": rng.normal(0, 1, d),\n            \"lr\": float(rng.choice([0.001, 0.01, 0.1])),\n            \"mu\": float(rng.choice([0.5, 0.9, 0.99]))}\n\ndef momentum_step(w, g, v, lr, mu):\n    w = np.asarray(w, dtype=float)\n    g = np.asarray(g, dtype=float)\n    v = np.asarray(v, dtype=float)\n    v_new = mu * v + g\n    return w - lr * v_new, v_new", "momentum_step", 20)

### Task 6 — Adam update step

●●● · ~30 min

> Adam maintains two exponential moving averages: m = b1*m + (1-b1)*g for the mean gradient and v = b2*v + (1-b2)*g^2 for its square. Since m and v start at zero they are biased toward zero in early steps — correct this with m_hat = m/(1-b1^t) and v_hat = v/(1-b2^t), then w <- w - lr * m_hat / (sqrt(v_hat) + eps).

Implement `adam_step(w, g, m, v, t, lr, b1, b2, eps)`: one Adam update at time step `t` (1-based, so t=1 is the first step). Update `m` and `v` first, apply bias correction, then update `w`. Return `(w_new, m_new, v_new)`.

<details><summary>Hint</summary>

m = b1*m + (1-b1)*g; v = b2*v + (1-b2)*g**2; mhat = m/(1-b1**t); vhat = v/(1-b2**t); w_new = w - lr*mhat/(np.sqrt(vhat)+eps).

</details>


In [ ]:
import numpy as np

def adam_step(w, g, m, v, t, lr, b1, b2, eps):
    w = np.asarray(w, dtype=float)
    g = np.asarray(g, dtype=float)
    m = np.asarray(m, dtype=float)
    v = np.asarray(v, dtype=float)
    # your code — plain SGD below ignores the moments, fix it
    return w - lr * g, m, v

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    d = int(rng.integers(2, 8))\n    return {\"w\": rng.normal(0, 1, d), \"g\": rng.normal(0, 1, d),\n            \"m\": rng.normal(0, 0.5, d), \"v\": rng.uniform(0, 0.5, d),\n            \"t\": int(rng.integers(1, 30)),\n            \"lr\": float(rng.choice([0.001, 0.01, 0.1])),\n            \"b1\": float(rng.choice([0.9, 0.95])),\n            \"b2\": float(rng.choice([0.99, 0.999])),\n            \"eps\": 1e-8}\n\ndef adam_step(w, g, m, v, t, lr, b1, b2, eps):\n    w = np.asarray(w, dtype=float)\n    g = np.asarray(g, dtype=float)\n    m = np.asarray(m, dtype=float)\n    v = np.asarray(v, dtype=float)\n    m = b1 * m + (1.0 - b1) * g\n    v = b2 * v + (1.0 - b2) * g ** 2\n    mhat = m / (1.0 - b1 ** t)\n    vhat = v / (1.0 - b2 ** t)\n    return w - lr * mhat / (np.sqrt(vhat) + eps), m, v", "adam_step", 20)

---

**Go deeper**
- [Original lesson (Microsoft, MIT)](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/3-NeuralNetworks/04-OwnFramework)
- [Yandex ML handbook (RU): neural networks](https://contest.yandex.ru/tracks/ml/deep-learning-introduction/neural-networks)
- [Yandex math handbook (RU): analysis and gradients](https://contest.yandex.ru/tracks/math/mathematical-analysis/chapter-overview)
- [Yandex math handbook (RU): linear algebra](https://contest.yandex.ru/tracks/math/linear-algebra/chapter-overview)
- [Yandex ML handbook (RU): optimization](https://contest.yandex.ru/tracks/ml/optimization-in-machine-learning/optimization-in-machine-learning)